In [199]:
from playwright.async_api import async_playwright

p = await async_playwright().start()
browser = await p.chromium.connect_over_cdp("http://127.0.0.1:9111")
context = await browser.new_context()
page = await context.new_page()

In [122]:
await page.goto("https://snoonu.com");

In [123]:
# hover the page
await page.hover("body")

In [124]:
search_results = page.locator('div[class*="SearchResults_group"]')

In [125]:
merchants = await search_results.locator(">div").all()

In [126]:
import json
from playwright.async_api import Route
captured_data = {}
page.set_default_navigation_timeout(0)
async def handle_route(route: Route):
    print("intercepting")
    response = await route.fetch()
    body = await response.body()
    data = json.loads(body.decode('utf-8'))
    captured_data['pageProps'] = data.get('pageProps', {})
    await route.fulfill(response=response)
await page.route("**/_next/data/**/search.json*", handle_route )


In [127]:
await page.goto("https://www.snoonu.com/snoonu-market/search?q=IPHONE%2017")


<Response url='https://www.snoonu.com/snoonu-market/search?q=IPHONE%2017' request=<Request url='https://www.snoonu.com/snoonu-market/search?q=IPHONE%2017' method='GET'>>

In [137]:
from urllib.parse import urlparse
from typing import TypedDict, Optional
from playwright.async_api import Locator, BrowserContext
import asyncio
results = []
hostname = urlparse(page.url).hostname
page.set_default_timeout(1000)

class item_result(TypedDict, total=False):
    name: Optional[str]
    image: Optional[str]
    price: Optional[str]
    url: Optional[str]
    
async def get_item_data(item: Locator):
    result: item_result = {}
    try:
        img = await item.locator("img").get_attribute("src")
        if (img and img.startswith("http")):
            result['image'] = img
    except:
        pass
    info = item.locator("div[class*='ProductCartVerticalDescription_info']")
    try:
        result['price'] = await info.locator("[class*='price']").text_content()
        result['name'] = await info.locator("[class*='name']").text_content()
        result['url'] = await item.locator("a").get_attribute("href")
    except:
        return
    return result


    
results = []
async def get_merchant_data(merchant: Locator):
    
    merchant_info = merchant.locator("div[class*='SearchMerchant_info']")
    merchant_name = await merchant_info.locator("[class*='SearchMerchant_name']").text_content()
    merchant_url = await merchant.locator(">a").get_attribute("href")
    merchant_carousel = merchant.locator("div[class*='SearchMerchant_carousel']")
    items = await merchant_carousel.locator(">div").all()
    
    print(f"Processing {merchant_name}")
    
    return {
        "name": merchant_name,
        "url": merchant_url,
        "items": await asyncio.gather(*
            [get_item_data(item) for item in items]
        , return_exceptions=False)}
    
for m in asyncio.as_completed((
        get_merchant_data(merchant)
        for merchant in await search_results.locator(">div").all()
)):
    result = await m
    if (result):
        results.append(result)

Processing Snoomart
Processing Maamel Arabic Coffee
Processing Ahlulkaif Coffee and Dates
Processing Luzina Nuts & Coffee
Processing Rouwad For Speciality Coffee
Processing Friends Coffee
Processing Kufiyeh Coffee
Processing Coffee Centre
Processing Al Mizaj Coffee
Processing Coffee 88
Processing Al Abed Roastery
Processing Nafahat Dirty Arabic Coffee
Processing Al Arrab Coffee & Roastery
Processing Soho Coffee CO
Processing Veeteek
Processing Meiano Coffee
Processing Gareeb Al-Gorba For Coffee
Processing Mastic Coffee
Processing Al Keef Al Jnoubi For Coffee And Date
Processing Alkafile Travel and Coffee


In [133]:
results[1]

{'name': 'Kufiyeh Coffee',
 'url': '/groceries/kufiyeh-coffee?source=global%20search',
 'items': []}

In [ ]:
hostname = urlparse(page.url).hostname

async def process_merchant(context: BrowserContext, merchant: Locator):
    page = await context.new_page()
    pass

for merchant in asyncio.as_completed(process_merchant(context, merchant) for merchant in merchants[:1]):
    data = await merchant
    print(data)

In [148]:
test_page = await context.new_page()

In [175]:
async def process_suggested_product(product: Locator):
    url = await product.get_attribute("href")
    name = await product.locator('[class*="ProductCardHorizontal_name"]').text_content()
    description = await product.locator('[class*="ProductCardHorizontal_description"]').text_content()
    price = await product.locator('[class*="priceWrapper"] > h5').text_content()
    discounted_price = product.locator('[class*="priceWrapper"] > p')
    if await discounted_price.is_visible(timeout=1000):
        discounted_price = await discounted_price.text_content()
    else:
        discounted_price = None
    
    return {
        'name': name,
        'description': description,
        'price': price,
        'url': url,
        'discounted_price': discounted_price
    }

async def process_merchant():
    result = results[0]
    # page = await context.new_page()
    # await test_page.goto(f"https://snoonu.com/{result['url']}")
    search_box = test_page.locator('[class*="SearchInMerchant_input"]')
    await search_box.fill("coffee")
    await search_box.press("Enter")
    suggested_products = test_page.locator('[class*="SuggestedProducts_group"]')
    await suggested_products.wait_for(state="visible")
    product_cards = await suggested_products.locator('[data-analytic-label*="productCard"]').all()
    for product in product_cards:
        result = await process_suggested_product(product)
        print(result)
    

await process_merchant()

{'name': 'Jordanian Coffee Plain', 'description': 'Traditional Jordanian Coffee Served Plain.', 'price': '22.5 QR', 'url': None, 'discounted_price': None}
{'name': 'Turkish Coffee Brazil With Cardamom', 'description': 'Turkish Coffee Made With Brazilian Beans And Cardamom.', 'price': '17.5 QR', 'url': None, 'discounted_price': None}
{'name': 'Turkish Coffee Ethiopian Plain', 'description': 'Turkish Coffee Made With Ethiopian Beans, Served Plain.', 'price': '22.5 QR', 'url': None, 'discounted_price': None}
{'name': 'Turkish Coffee Colombian Plain', 'description': 'Turkish Coffee Made With Colombian Beans, Served Plain.', 'price': '20 QR', 'url': None, 'discounted_price': None}
{'name': 'Authentic Coffee Beans Colombian', 'description': 'Authentic Colombian Coffee Beans With A Rich And Bold Taste.', 'price': '20 QR', 'url': None, 'discounted_price': None}
{'name': 'Aroma Brazilian Coffee Beans', 'description': 'Aromatic Brazilian Coffee Beans With A Smooth And Nutty Flavor.', 'price': '1

In [203]:

# page = await context.new_page()
# await page.goto(f"https://snoonu.com/{result['url']}")
search_box = page.locator('[class*="SearchInMerchant_input"]')
await search_box.fill("coffee")
await search_box.press("Enter")
suggested_products = page.locator('[class*="SuggestedProducts_group"]')
await suggested_products.wait_for(state="visible")
product_cards = await suggested_products.locator('[data-analytic-label*="productCard"]').all()


In [ ]:
for product in product_cards:
    url = await product.get_attribute("href")
    name = await product.locator('[class*="ProductCardHorizontal_name"]').text_content()
    print(name)
    description = await product.locator('[class*="ProductCardHorizontal_description"]').text_content()
    discount_wrapper = product.locator('[class*="priceWrapper"]')
    if await discount_wrapper.is_visible(timeout=2000):
        price = await discount_wrapper.locator('[class*="ProductCardHorizontal_oldPrice"]').text_content()
        discounted_price = await discount_wrapper.locator('[class*="ProductCardHorizontal_newPrice"]').text_content()

TimeoutError: Locator.text_content: Timeout 30000ms exceeded.
Call log:
  - waiting for locator("[class*=\"SuggestedProducts_group\"]").locator("[data-analytic-label*=\"productCard\"]").first.locator("[class*=\"priceWrapper\"]").locator("[class*=\"ProductCardHorizontal_oldPrice\"]")


In [192]:
await page.locator("[class*='newPrice']").all()

TargetClosedError: Locator.all: Target page, context or browser has been closed